## Testing

In [ ]:
import json
import math
import numpy as np
from pathlib import Path
from scipy.sparse import load_npz, lil_matrix, csr_matrix

Test set A and B BM25 score - average document length and term frequency from training corpus

In [ ]:

matrices_folder = "../NMF_code/articles_raw_proc_data/matrices"
data_folder     = "../articles_raw_proc_data/processed_data"
field           = "text_all_lemma"

k1 = 1.5
b  = 0.75

q1_with_dupes = [
    "bank", "banker", "banking", "fine", "financial", "executive",
    "regulator", "chief", "institution", "scandal", "deposit"
]
q1 = []
for word in q1_with_dupes:
    if word not in q1:
        q1.append(word)

q2_extras_with_dupes = [
    "settle", "settlement", "justice", "prosecution", "lawsuit",
    "sanction", "rig", "rigging", "compliance"
]
q2_extras = []
for word in q2_extras_with_dupes:
    if word not in q2_extras:
        q2_extras.append(word)

q2 = []
for word in q1:
    q2.append(word)
for word in q2_extras:
    if word not in q2:
        q2.append(word)

query_terms = q2
print("scoring with " + str(len(query_terms)) + " query terms")


#LOAD TRAINING VOCABULARY AND IDF STATISTICS
------

dtm_train_path = matrices_folder + "/dtm_"   + field + ".npz"
vocab_path     = matrices_folder + "/vocab_" + field + ".npy"

print("loading training dtm from: " + dtm_train_path)
dtm_train = load_npz(dtm_train_path)

print("loading vocabulary from: " + vocab_path)
vocab = np.load(vocab_path, allow_pickle=True)

#build term -> column index lookup
term2idx = {}
idx = 0
for term in vocab:
    term2idx[term] = idx
    idx = idx + 1

vocab_size = len(term2idx)
print("vocabulary size: " + str(vocab_size))

n_train = dtm_train.shape[0]
print("number of training documents: " + str(n_train))

#document frequency per term — from training only
df_counts = np.array((dtm_train > 0).sum(axis=0)).ravel()

#average document length from training set
dls_train = np.array(dtm_train.sum(axis=1)).ravel()
total_train_tokens = 0.0
for dl in dls_train:
    total_train_tokens = total_train_tokens + dl
avgdl = total_train_tokens / n_train
print("training avgdl: " + str(round(avgdl, 2)))

#---------------------------------------------------------------------------
#PROCESS BOTH TEST SPLITS
#---------------------------------------------------------------------------

split_names = ["test_a", "test_b"]

for split_index in range(len(split_names)):
    split     = split_names[split_index]
    file_path = data_folder + "/guardian_" + split + ".json"

    print("")
    print("processing: " + file_path)

    #load the test documents
    f = open(file_path, "r", encoding="utf-8")
    docs = json.load(f)
    f.close()

    n_docs = len(docs)
    print("loaded " + str(n_docs) + " documents")

    
    #STEP 1: BUILD THE DTM FOR THIS TEST SPLIT
    #rows = documents, columns = vocabulary terms (same columns as train dtm)
    #we use lil_matrix for building because it's efficient for row-by-row inserts
 
    print("building dtm for " + split + " ...")

    dtm_test = lil_matrix((n_docs, vocab_size), dtype=np.float32)

    for doc_index in range(n_docs):
        doc  = docs[doc_index]
        text = doc.get(field, "")

        #tokenise and count — only track terms in the training vocabulary
        tokens = text.split()
        for tok in tokens:
            if tok in term2idx:
                col = term2idx[tok]
                dtm_test[doc_index, col] = dtm_test[doc_index, col] + 1

        #progress print every 5000 documents
        if doc_index % 5000 == 0 and doc_index > 0:
            print("  built " + str(doc_index) + " rows so far...")

    #convert to csr for fast column access during scoring
    dtm_test_csr = csr_matrix(dtm_test)
    print("dtm built, shape: " + str(dtm_test_csr.shape))

    #document lengths from the test dtm (row sums)
    dls_test = np.array(dtm_test_csr.sum(axis=1)).ravel()


    #STEP 2: SCORE EACH DOCUMENT USING THE TEST DTM
    #same bm25 formula as training — idf from train, tf from test dtm
   

    print("scoring documents...")

    #start with all zeros
    scores = np.zeros(n_docs)

    for term in query_terms:
        #skip terms not in vocabulary
        if term not in term2idx:
            print("  skipping term not in vocab: " + term)
            continue

        col  = term2idx[term]
        df_t = df_counts[col]  #document frequency from TRAINING set

        #idf from training statistics
        idf_numerator   = n_train - df_t + 0.5
        idf_denominator = df_t + 0.5
        idf = math.log((idf_numerator / idf_denominator) + 1)

        #term frequencies from the TEST dtm
        tf_raw = np.array(dtm_test_csr[:, col].todense()).ravel()

        #normalised tf using test document lengths but training avgdl
        length_norm    = 1 - b + b * (dls_test / avgdl)
        tf_numerator   = tf_raw * (k1 + 1)
        tf_denominator = tf_raw + k1 * length_norm
        tf_norm        = tf_numerator / tf_denominator

        scores = scores + idf * tf_norm

    print("scoring done")

   
    #STEP 3: ATTACH SCORES TO DOCUMENTS AND SAVE
   

    for doc_index in range(n_docs):
        docs[doc_index]["bm25_q2"] = round(float(scores[doc_index]), 4)

    f = open(file_path, "w", encoding="utf-8")
    json.dump(docs, f, ensure_ascii=False, indent=2)
    f.close()
    print("saved to: " + file_path)

    #summary statistics
    nz_list = []
    for s in scores:
        if s > 0:
            nz_list.append(s)
    nz = np.array(nz_list)

    nz_total = 0.0
    for s in nz:
        nz_total = nz_total + s
    nz_mean = nz_total / len(nz)

    nz_max = nz[0]
    for s in nz:
        if s > nz_max:
            nz_max = s

    filename = file_path.split("/")[-1]
    print(filename + ": n=" + str(n_docs) +
          "  non-zero=" + str(len(nz)) +
          "  mean=" + str(round(nz_mean, 2)) +
          "  max="  + str(round(nz_max,  2)))

Applying threshold to test set A and test set B - code below for test set A

In [ ]:

#the threshold we computed from the mixture model f1 sweep on the training set
threshold = 16.0876

#path to the test set — plain string
data_folder  = "../articles_raw_proc_data/processed_data"
test_a_path  = data_folder + "/guardian_test_a.json"

#LOAD TEST SET A


print("loading test set A from: " + test_a_path)
f = open(test_a_path, "r", encoding="utf-8")
docs = json.load(f)
f.close()
print("loaded " + str(len(docs)) + " documents")

#collect scores into a plain list then convert to numpy array
scores_list = []
for d in docs:
    scores_list.append(d["bm25_q2"])
scores = np.array(scores_list)

-----------
#SPLIT INTO RELEVANT (above threshold) AND NON-RELEVANT (below threshold)

relevant     = []
non_relevant = []

for d in docs:
    if d["bm25_q2"] >= threshold:
        relevant.append(d)
    else:
        non_relevant.append(d)

print("Retrieved (relevant):  " + f"{len(relevant):,}")
print("Not retrieved:         " + f"{len(non_relevant):,}")


#TOP 10 DOCUMENTS BY SCORE

#make a copy of docs so we don't disturb the original list
docs_to_sort = []
for d in docs:
    docs_to_sort.append(d)

top10 = []
for _ in range(10):
    if len(docs_to_sort) == 0:
        break
    #find the document with the highest score in the remaining list
    best_pos   = 0
    best_score = docs_to_sort[0]["bm25_q2"]
    for pos in range(len(docs_to_sort)):
        if docs_to_sort[pos]["bm25_q2"] > best_score:
            best_score = docs_to_sort[pos]["bm25_q2"]
            best_pos   = pos
    top10.append(docs_to_sort[best_pos])
    docs_to_sort.pop(best_pos)

print("")
print("Top 10 by score:")
for d in top10:
    score_str = f"{d['bm25_q2']:.2f}"  #keeping f-string, float formatting is awkward otherwise
    print("  " + score_str + "  " + str(d["id"]))

#
#5 DOCUMENTS EITHER SIDE OF THE THRESHOLD

#full ascending sort using selection sort
#i know this is slow but it makes the logic clear
print("")
print("sorting all documents by score to find threshold boundary...")

docs_for_sort = []
for d in docs:
    docs_for_sort.append(d)

sorted_docs_asc = []
counter = 0
while len(docs_for_sort) > 0:
    #find the document with the LOWEST score this time (ascending sort)
    lowest_pos   = 0
    lowest_score = docs_for_sort[0]["bm25_q2"]
    for pos in range(len(docs_for_sort)):
        if docs_for_sort[pos]["bm25_q2"] < lowest_score:
            lowest_score = docs_for_sort[pos]["bm25_q2"]
            lowest_pos   = pos
    sorted_docs_asc.append(docs_for_sort[lowest_pos])
    docs_for_sort.pop(lowest_pos)
    counter = counter + 1
    if counter % 5000 == 0:
        print("  sorted " + str(counter) + " documents so far...")

print("sorting done")

#find the index of the first document at or above the threshold
#this is where the boundary sits in the sorted list
thresh_idx = None
for i in range(len(sorted_docs_asc)):
    if sorted_docs_asc[i]["bm25_q2"] >= threshold:
        thresh_idx = i
        break

#handle edge case where no document reaches the threshold
if thresh_idx is None:
    print("WARNING: no document scored at or above threshold " + str(threshold))
else:
    #5 just below: indices thresh_idx-5 to thresh_idx-1
    #making sure we don't go below index 0
    below_start = thresh_idx - 5
    if below_start < 0:
        below_start = 0

    just_below = []
    for i in range(below_start, thresh_idx):
        just_below.append(sorted_docs_asc[i])

    #5 just above: indices thresh_idx to thresh_idx+4
    just_above = []
    for i in range(thresh_idx, thresh_idx + 5):
        if i < len(sorted_docs_asc):  #don't go past the end of the list
            just_above.append(sorted_docs_asc[i])

    #print just_below in descending order (closest to threshold first)
    #reversing a list manually by going backwards through the indices
    print("5 just below threshold:")
    for i in range(len(just_below) - 1, -1, -1):
        d = just_below[i]
        score_str = f"{d['bm25_q2']:.2f}"
        print("  " + score_str + "  " + str(d["id"]))

    #just_above is already in ascending order (lowest first)
    print("")
    print("5 just above threshold:")
    for d in just_above:
        score_str = f"{d['bm25_q2']:.2f}"
        print("  " + score_str + "  " + str(d["id"]))


#HISTOGRAM OF NON-ZERO SCORES WITH THRESHOLD LINE

#collect non-zero scores manually
nonzero_list = []
for s in scores:
    if s > 0:
        nonzero_list.append(s)
nonzero = np.array(nonzero_list)

print("")
print("plotting score distribution...")

fig = go.Figure()

#histogram of non-zero scores
fig.add_trace(
    go.Histogram(
        x=nonzero,
        nbinsx=80,
        marker_color="#2166ac",
        opacity=0.6
    )
)

#vertical line marking the threshold
fig.add_vline(
    x=threshold,
    line_color="red",
    line_dash="dash",
    annotation_text="threshold = " + str(threshold),
    annotation_position="top right"
)

fig.update_layout(
    title="BM25 Q2 score distribution — Test set A",
    xaxis_title="BM25 score",
    yaxis_title="count",
    width=900,
    height=450
)

fig.show()